# *Avocado ML Regressor for Nextmv Workflow*

This notebook shows how you can utilize Databricks to push and run a model on Nextmv. 

Why use Nextmv behind the scenes?
* View and share your results via the Nextmv UI
* Manage and compare multiple versions of your model
* Provide a no-code UI to others to test model parameters

## Prerequisites 
To run this notebook example you'll need to have your Nextmv API Key as a managed secret. 

```
databricks secrets put-secret --json '{
        "scope": "<scope-name>",
        "key": "nextmv-api-key",
        "string_value": "<api-key-secret>"
}
```


In [0]:
### pip install your requirments

%pip install --upgrade "nextmv[all]"
%pip install nextmv-gurobipy
%pip install gurobipy
%pip install plotly

In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
%python
api_key = dbutils.secrets.get(scope="my-scope", key="nextmv-api-key")

In [0]:
## Import libraries

import json

import nextmv
import pandas as pd
import statsmodels.formula.api as smf
from nextmv import cloud
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [0]:
avocado = pd.read_csv(
    "https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/HABdata_2019_2022.csv"
)  # dataset downloaded directly from HAB
avocado_old = pd.read_csv(
    "https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/kaggledata_till2018.csv"
)  # dataset downloaded from Kaggle
avocado = pd.concat([avocado, avocado_old], ignore_index=True)

# Add the index for each year from 2015 through 2022
avocado["date"] = pd.to_datetime(avocado["date"])
avocado["year"] = pd.DatetimeIndex(avocado["date"]).year
avocado["year_index"] = avocado["year"] - 2015
avocado = avocado.sort_values(by="date")

# Define the peak season
avocado["month"] = pd.DatetimeIndex(avocado["date"]).month
peak_months = range(2, 8)


def peak_season(row):
    return 1 if int(row["month"]) in peak_months else 0


avocado["peak"] = avocado.apply(lambda row: peak_season(row), axis=1)

# Scale the number of avocados to millions
avocado["units_sold"] = avocado["units_sold"] / 1000000

# Select only conventional avocados
avocado = avocado[avocado["type"] == "Conventional"]

avocado = avocado[["date", "units_sold", "price", "region", "year", "month", "year_index", "peak"]].reset_index(
    drop=True
)
regions = [
    "Great_Lakes",
    "Midsouth",
    "Northeast",
    "Northern_New_England",
    "SouthCentral",
    "Southeast",
    "West",
    "Plains",
]
df = avocado[avocado.region.isin(regions)]

for col in df.select_dtypes(include=["datetime"]).columns:
    df[col] = df[col].astype(str)

In [0]:
input = nextmv.Input(data=df.to_dict())
df = pd.DataFrame(input.data)
# <<<<<<<<<<<<< Stop Nextmv-ifying
train, test = train_test_split(df, train_size=0.8, random_state=1)
df_train = pd.DataFrame(train, columns=df.columns)
df_test = pd.DataFrame(test, columns=df.columns)

# Train the model
formula = "units_sold ~ price + year_index + C(region)+ peak"
mod = smf.ols(formula, data=df_train)
result = mod.fit()
result.summary()

# Get R^2 from test data
y_true = df_test["units_sold"]
y_pred = result.predict(df_test)

formula = "units_sold ~ price + year_index + C(region)+ peak"
mod_full = smf.ols(formula, data=df)
result_full = mod_full.fit()

y_true_full = df["units_sold"]
y_pred_full = result_full.predict(df)

# Get the weights and store it
coef_dict = result_full.params.to_dict()
coef_dict["C(region)[T.Great_Lakes]"] = 0

# >>>>>>>>>>>> Start Nextmv-ifying
statistics = nextmv.Statistics(
    result=nextmv.ResultStatistics(
        custom={
            "r2_test": r2_score(y_true, y_pred),
            "r2_full": r2_score(y_true_full, y_pred_full),
        },
    ),
)

data = {
    "regions": [
        "Great_Lakes",
        "Midsouth",
        "Northeast",
        "Northern_New_England",
        "SouthCentral",
        "Southeast",
        "West",
        "Plains",
    ],
    "total_amount_of_supply": 30,
    "cost_per_wasted_product": 0.1,
    "peak": 1,
    "transport_costs": {
        "Great_Lakes": 0.3,
        "Midsouth": 0.1,
        "Northeast": 0.4,
        "Northern_New_England": 0.5,
        "SouthCentral": 0.3,
        "Southeast": 0.2,
        "West": 0.2,
        "Plains": 0.2,
    },
    "year": 2022,
    "minimum_product_price": 0,
    "maximum_product_price": 2,
    "minimum_product_allocations": dict(df.groupby("region")["units_sold"].min()),
    "maximum_product_allocations": dict(df.groupby("region")["units_sold"].max()),
    "coefficients": coef_dict,
}


output = nextmv.Output(
    options=nextmv.Options(),
    solution=data,
    statistics=statistics,
)

In [0]:
print(json.dumps(output.solution, indent=2))

In [0]:
### Using your api key, connect to Nextmv

client = cloud.Client(api_key=api_key)

In [0]:
## Create the avocado-ml-regressor app in my account if it doesn't exist in your Nextmv account.

app_name = "avocado-ml-regressor"
app = cloud.Application.new(client=client, id=app_name, name=app_name, exist_ok=True)

In [0]:
# Start tracking runs
result = app.track_run_with_result(
    tracked_run=cloud.TrackedRun(
        input=input,
        output=output,
        status=cloud.TrackedRunStatus.SUCCEEDED,
        duration=1,
    )
)
print(f"Run {result.id} tracked successfully.")

In [0]:
result.output["statistics"]["result"]["custom"]["run_id"] = result.id
result.output["statistics"]["result"]["custom"]["app_id"] = app.id

In [0]:
# Output from the notebook that will surface in the workflow app on Nextmv

dbutils.notebook.exit(json.dumps(result.output))

Visit the `Apps + Workflows` space on [https://cloud.nextmv.io/apps](https://cloud.nextmv.io/apps) to view results in your app.